# STEP 5 – Fine-Tuning IndoBERT
Fine-tuning `indobenchmark/indobert-base-p1` untuk klasifikasi 3 kelas:  
`normal`, `promo`, `penipuan`.

## 1. Instalasi Library

Jalankan cell ini sekali di awal sesi Colab. Library yang dibutuhkan:
- `transformers` — menyediakan model dan tokenizer IndoBERT dari Hugging Face
- `accelerate` — dibutuhkan oleh `transformers` untuk training yang lebih stabil

Library lainnya (`torch`, `sklearn`, `pandas`, `matplotlib`, `seaborn`) sudah tersedia secara default di Colab.

In [4]:
# Install dependencies
!pip install transformers accelerate -q


## 2. Mount Google Drive

Google Drive di-mount agar notebook bisa membaca file dataset (`train.csv`, `val.csv`, `test.csv`, `class_weights.json`) dan menyimpan model hasil training.

Pastikan folder `sms-spam-indonesia/` sudah ada di `MyDrive` sebelum menjalankan cell konfigurasi di bawah. Struktur yang diharapkan:
```
MyDrive/
  sms-spam-indonesia/
    train.csv
    val.csv
    test.csv
    class_weights.json
```

In [5]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 3. Import Library

Semua dependensi diimpor di sini. Beberapa catatan penting:
- `AutoTokenizer` dan `AutoModelForSequenceClassification` digunakan untuk memuat model IndoBERT beserta tokenizer-nya secara otomatis dari Hugging Face.
- `get_linear_schedule_with_warmup` — scheduler learning rate yang menurunkan LR secara linear setelah fase warmup.
- `AdamW` — optimizer Adam dengan weight decay, standar untuk fine-tuning model transformer.
- `matplotlib` dan `seaborn` dipakai untuk visualisasi confusion matrix dan training curve.

In [6]:
import os
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns


## Konfigurasi

In [7]:
# Mount Google Drive terlebih dahulu (jalankan cell mount di bawah sebelum ini)
MODEL_NAME   = "indobenchmark/indobert-base-p1"
TRAIN_PATH   = "/content/drive/MyDrive/sms-spam-indonesia/train.csv"
VAL_PATH     = "/content/drive/MyDrive/sms-spam-indonesia/val.csv"
TEST_PATH    = "/content/drive/MyDrive/sms-spam-indonesia/test.csv"
WEIGHTS_PATH = "/content/drive/MyDrive/sms-spam-indonesia/class_weights.json"
OUTPUT_DIR   = "/content/drive/MyDrive/indobert"

MAX_LEN      = 128
BATCH_SIZE   = 16
EPOCHS       = 3
LR           = 2e-5
WEIGHT_DECAY = 0.01
RANDOM_STATE = 42

LABEL2ID = {"normal": 0, "promo": 1, "penipuan": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR siap:", OUTPUT_DIR)


OUTPUT_DIR siap: /content/drive/MyDrive/indobert


### Penjelasan Parameter

| Parameter | Nilai | Keterangan |
|---|---|---|
| `MAX_LEN` | 128 | Panjang maksimum token per input. SMS umumnya pendek, sehingga 128 token sudah cukup dan efisien secara komputasi. |
| `BATCH_SIZE` | 16 | Jumlah sampel per iterasi training. Nilai 16 dipilih untuk keseimbangan antara kecepatan dan stabilitas gradien di GPU Colab. |
| `EPOCHS` | 3 | Jumlah epoch training. Fine-tuning model BERT umumnya cukup dengan 3–5 epoch untuk menghindari overfitting. |
| `LR` | 2e-5 | Learning rate kecil yang umum dipakai untuk fine-tuning transformer agar bobot pretrained tidak rusak. |
| `WEIGHT_DECAY` | 0.01 | Regularisasi L2 pada AdamW untuk mencegah overfitting. |

Label mapping: `normal=0`, `promo=1`, `penipuan=2`.

## Dataset

In [8]:
class SMSDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts     = df["text"].tolist()
        self.labels    = df["label"].map(LABEL2ID).tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }

### Penjelasan SMSDataset

`SMSDataset` adalah custom PyTorch `Dataset` yang membungkus data CSV menjadi format yang siap diproses model.

Setiap item yang dikembalikan `__getitem__` berisi tiga tensor:
- `input_ids` — representasi numerik token dari teks SMS setelah tokenisasi.
- `attention_mask` — mask biner (1/0) yang menandai token asli vs token padding, sehingga model tidak memproses padding sebagai informasi nyata.
- `label` — kelas target dalam bentuk integer (0, 1, atau 2).

Parameter `padding='max_length'` dan `truncation=True` memastikan semua input memiliki panjang seragam (`MAX_LEN=128`).

## Training & Evaluasi (fungsi helper)

In [9]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, total_correct = 0, 0

    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss    += loss.item()
        total_correct += (outputs.logits.argmax(dim=1) == labels).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = total_correct / len(loader.dataset)
    return avg_loss, accuracy


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct = 0, 0

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = criterion(outputs.logits, labels)

            total_loss    += loss.item()
            total_correct += (outputs.logits.argmax(dim=1) == labels).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = total_correct / len(loader.dataset)
    return avg_loss, accuracy

### Penjelasan Fungsi Training

**`train_epoch`** menjalankan satu epoch training penuh:
1. Model diset ke mode `train()` agar layer seperti dropout aktif.
2. Setiap batch diproses: hitung loss, backpropagation, update bobot.
3. `clip_grad_norm_` membatasi gradien maksimum di 1.0 untuk mencegah *exploding gradient*.
4. `scheduler.step()` dipanggil setelah setiap batch untuk update learning rate.

**`eval_epoch`** menjalankan evaluasi tanpa update bobot:
1. Model diset ke mode `eval()` agar dropout nonaktif.
2. `torch.no_grad()` menonaktifkan komputasi gradien untuk menghemat memori.
3. Loss dan akurasi dihitung untuk memantau kondisi model di validation set.

In [10]:
def evaluate_test(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = outputs.logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return all_labels, all_preds


def save_confusion_matrix(labels, preds, output_dir):
    cm = confusion_matrix(labels, preds)
    label_names = [ID2LABEL[i] for i in range(len(ID2LABEL))]

    plt.figure(figsize=(7, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=label_names, yticklabels=label_names
    )
    plt.title("Confusion Matrix - IndoBERT")
    plt.ylabel("Label Sebenarnya")
    plt.xlabel("Label Prediksi")
    plt.tight_layout()
    path = os.path.join(output_dir, "confusion_matrix.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Confusion matrix disimpan: {path}")


def save_training_curve(history, output_dir):
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("Training Curve - IndoBERT", fontsize=13, fontweight="bold")

    axes[0].plot(epochs, history["train_loss"], label="Train")
    axes[0].plot(epochs, history["val_loss"],   label="Val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], label="Train")
    axes[1].plot(epochs, history["val_acc"],   label="Val")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()

    plt.tight_layout()
    path = os.path.join(output_dir, "training_curve.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Training curve disimpan: {path}")

### Penjelasan Fungsi Evaluasi & Visualisasi

**`evaluate_test`** mengumpulkan semua prediksi model pada test set tanpa menghitung loss, hanya untuk mendapatkan label prediksi final.

**`save_confusion_matrix`** menyimpan heatmap confusion matrix ke file PNG. Confusion matrix memperlihatkan seberapa sering model salah mengklasifikasikan satu kelas sebagai kelas lain, berguna untuk analisis error per kelas.

**`save_training_curve`** menyimpan grafik loss dan akurasi training vs validation per epoch. Grafik ini dipakai untuk mendeteksi overfitting (val loss naik sementara train loss turun) atau underfitting (keduanya masih tinggi).

## Pipeline Utama

In [11]:
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load data
df_train = pd.read_csv(TRAIN_PATH)
df_val   = pd.read_csv(VAL_PATH)
df_test  = pd.read_csv(TEST_PATH)
print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

# Load class weights
with open(WEIGHTS_PATH) as f:
    weights_dict = json.load(f)
weights_tensor = torch.tensor(
    [weights_dict[ID2LABEL[i]] for i in range(len(ID2LABEL))],
    dtype=torch.float
).to(device)
print(f"Class weights: {weights_dict}")

Device: cuda
Train: 799 | Val: 229 | Test: 115
Class weights: {'normal': 0.669179229480737, 'penipuan': 1.1381766381766383, 'promo': 1.594810379241517}


### Langkah 1 — Load Data & Class Weights

Data training, validation, dan test dimuat dari Google Drive. `class_weights.json` berisi bobot per kelas yang telah dihitung sebelumnya menggunakan `sklearn.utils.class_weight.compute_class_weight` dengan parameter `balanced`.

Bobot ini akan digunakan di fungsi loss (`CrossEntropyLoss`) agar model tidak bias ke kelas mayoritas akibat ketidakseimbangan distribusi kelas pada dataset.

In [12]:
print(f"\nMemuat tokenizer dan model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to(device)


Memuat tokenizer dan model: indobenchmark/indobert-base-p1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Langkah 2 — Muat Tokenizer & Model

`AutoTokenizer` memuat tokenizer IndoBERT yang sudah dilatih khusus untuk bahasa Indonesia, termasuk kosakata formal, informal, dan singkatan umum.

`AutoModelForSequenceClassification` memuat bobot pretrained IndoBERT dan secara otomatis menambahkan *classification head* (linear layer) di atas encoder untuk tugas 3 kelas. Parameter `id2label` dan `label2id` disimpan di konfigurasi model agar label dapat dibaca langsung saat inference.

In [13]:
train_dataset = SMSDataset(df_train, tokenizer, MAX_LEN)
val_dataset   = SMSDataset(df_val,   tokenizer, MAX_LEN)
test_dataset  = SMSDataset(df_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

### Langkah 3 — Buat DataLoader

`DataLoader` membungkus dataset menjadi iterator yang menyajikan data dalam batch. `shuffle=True` pada `train_loader` memastikan urutan data diacak setiap epoch untuk mengurangi bias urutan saat training. Validation dan test loader tidak diacak agar hasil evaluasi konsisten.

In [14]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)
criterion = torch.nn.CrossEntropyLoss(weight=weights_tensor)

### Langkah 4 — Optimizer, Scheduler & Loss

- **AdamW** dipilih karena menggabungkan adaptive learning rate (Adam) dengan weight decay yang terpisah dari gradien, terbukti lebih stabil untuk fine-tuning transformer dibanding Adam biasa.
- **Linear warmup scheduler**: learning rate dinaikkan bertahap selama 10% langkah pertama (*warmup*), lalu diturunkan secara linear hingga 0 di akhir training. Ini mencegah update yang terlalu besar di awal training yang dapat merusak bobot pretrained.
- **CrossEntropyLoss dengan class weights**: setiap kesalahan prediksi pada kelas minoritas diberi penalti lebih besar, mendorong model untuk lebih memperhatikan kelas yang jarang muncul.

## Training Loop

In [15]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    val_loss,   val_acc   = eval_epoch(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    # Simpan model terbaik berdasarkan val loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"  Model terbaik disimpan (val_loss: {val_loss:.4f})")

Epoch 1/3 | Train Loss: 0.4769 Acc: 0.8173 | Val Loss: 0.1133 Acc: 0.9825


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Model terbaik disimpan (val_loss: 0.1133)
Epoch 2/3 | Train Loss: 0.1591 Acc: 0.9574 | Val Loss: 0.1302 Acc: 0.9651
Epoch 3/3 | Train Loss: 0.0772 Acc: 0.9812 | Val Loss: 0.0869 Acc: 0.9869


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Model terbaik disimpan (val_loss: 0.0869)


### Penjelasan Training Loop

Setiap epoch, model dilatih pada training set lalu langsung dievaluasi pada validation set. Metrik loss dan akurasi dicatat di `history` untuk keperluan plotting.

Model hanya disimpan jika `val_loss` epoch sekarang lebih kecil dari yang terbaik sebelumnya (*best checkpoint*). Strategi ini mencegah penyimpanan model yang sudah mulai overfit.

## Evaluasi Test Set

In [16]:
print("Evaluasi pada test set...")
model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR).to(device)
all_labels, all_preds = evaluate_test(model, test_loader, device)

label_names = [ID2LABEL[i] for i in range(len(ID2LABEL))]
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=label_names))
print(f"Accuracy: {accuracy_score(all_labels, all_preds):.4f}")

# Simpan hasil evaluasi
report = classification_report(all_labels, all_preds, target_names=label_names, output_dict=True)
with open(os.path.join(OUTPUT_DIR, "eval_results.json"), "w") as f:
    json.dump(report, f, indent=2)

save_confusion_matrix(all_labels, all_preds, OUTPUT_DIR)
save_training_curve(history, OUTPUT_DIR)

print(f"\nSelesai. Semua output disimpan di: {OUTPUT_DIR}")

Evaluasi pada test set...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Classification Report:
              precision    recall  f1-score   support

      normal       0.98      1.00      0.99        57
       promo       0.89      1.00      0.94        24
    penipuan       1.00      0.88      0.94        34

    accuracy                           0.97       115
   macro avg       0.96      0.96      0.96       115
weighted avg       0.97      0.97      0.96       115

Accuracy: 0.9652
Confusion matrix disimpan: /content/drive/MyDrive/indobert/confusion_matrix.png
Training curve disimpan: /content/drive/MyDrive/indobert/training_curve.png

Selesai. Semua output disimpan di: /content/drive/MyDrive/indobert


### Penjelasan Evaluasi Akhir

Model terbaik (berdasarkan val loss terendah) dimuat ulang dari disk untuk evaluasi final pada test set yang belum pernah dilihat model selama training.

Hasil evaluasi mencakup:
- **Classification report** — precision, recall, dan F1-score per kelas (normal, promo, penipuan) beserta rata-rata macro dan weighted.
- **Accuracy** — persentase prediksi benar secara keseluruhan.
- **Confusion matrix** — disimpan sebagai `confusion_matrix.png`.
- **Training curve** — disimpan sebagai `training_curve.png`.
- **eval_results.json** — seluruh hasil classification report dalam format JSON untuk keperluan perbandingan dengan model XLM-RoBERTa nantinya.